# M2 — Where is difficulty represented inside the 7B PRM, and can it be removed?

The experiment that earns the mech-interp intersection. On **Kaggle T4×2, FULL PRECISION (fp16, no
quantization)** — the 7B shards across both GPUs via `device_map="auto"`.

Pipeline:
1. **Capture** residual-stream activations (every 4th layer) at each step's tag token, over the
   1,700 solutions. One forward pass also gives the PRM step scores (baseline).
2. **Probe** each layer: linear probe -> log response length & n_steps. R²-by-layer = *where
   difficulty lives* (a figure).
3. **Ablate** at the peak layer: MEAN-ablate the length direction from the residual stream (not
   zero-ablate). Re-score with the hook live.
4. **Re-measure** raw/ctl f1_B + the step-label gate on the ablated model.

**Decisive comparison (pre-commit to reporting whichever occurs):**
- raw f1_B drops toward its controlled value → confound is a localizable, removable direction (best).
- raw drops AND the step gate drops → removed competence, not just confound (weaker, still informative).
- raw barely moves → difficulty is distributed, not one direction (real negative, reportable).

### Precision note (read before comparing to the paper)
T4 is Turing → **fp16**, while the paper's Math-Shepherd numbers are **bf16** on A100. fp16 is full
16-bit precision, NOT quantization — but to stay apples-to-apples the notebook **recomputes the
fp16 baseline here** and compares ablated-vs-baseline in the SAME precision. Do not compare the
ablated fp16 number directly to the paper's bf16 0.075.

### Kaggle setup
- Accelerator: **GPU T4 x2**. Internet: **ON** (downloads Math-Shepherd ~14 GB).
- Attach a dataset with `step_cache.pt` + `candidates.jsonl`; set `INP` below.
- Session wall is 9h; capture(~45m)+ablate(~45m) fits. Activations are saved to
  `/kaggle/working` so Stage C can re-run without re-capturing.


In [ ]:
INP   = "/kaggle/input/ridae-m1"                       # dataset with step_cache.pt + candidates.jsonl
MODEL = "peiyi9979/math-shepherd-mistral-7b-prm"
EVERY = 4                                               # cache every 4th layer
WORK  = "/kaggle/working"
import torch
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "need T4 x2 for full-precision 7B"

In [ ]:
# Load Math-Shepherd in fp16, sharded across both T4s. No quantization.
from transformers import AutoTokenizer, AutoModelForCausalLM
# use_fast=False -> the SentencePiece tokenizer the A100 run used (clean 2-token candidate set
# [648,387]); the FAST tokenizer splits "+ -" into 3 tokens and corrupts scoring.
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)
CAND = tok.encode("+ -")[1:]            # expect exactly [id('+'), id('-')]
assert len(CAND) == 2, f"expected 2 candidate tokens, got {CAND} -- tokenizer mismatch"
TAG  = tok.encode("ки")[-1]
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             device_map="auto", output_hidden_states=True).eval()
NL = model.config.num_hidden_layers
LAYERS = list(range(0, NL+1, EVERY))    # indices into hidden_states (0=embeddings)
print(f"layers={NL}, caching hidden_states at {LAYERS}, cand ids {CAND}, tag {TAG}")

In [ ]:
import json, numpy as np, glob
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input — attach the dataset (Add Input)"
    return hits[0]
SC, CJ = find("step_cache.pt"), find("candidates.jsonl"); print("using:", SC, "|", CJ)
recs = torch.load(SC, weights_only=False)
meta = {json.loads(l)['record_id']: json.loads(l) for l in open(CJ) if l.strip()}
y = np.array([r['chain'] for r in recs])
# confound targets + matrix
L,NS,LA,DS=[],[],[],[]
for r in recs:
    m=meta.get(r['id'],{}); t=m.get('response_text') or m.get('full_text') or ""
    L.append(np.log1p(len(t.split()))); NS.append(len(r['steps_text']))
    LA.append((t.count(chr(92))+t.count('$'))/max(len(t.split()),1)); DS.append(r['split'])
L=np.array(L); NS=np.array(NS,float); LA=np.array(LA); DS=np.array(DS)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
surf=StandardScaler().fit_transform(np.c_[L,LA,NS])
oh=OneHotEncoder(sparse_output=False,handle_unknown='ignore').fit_transform(DS.reshape(-1,1))
CF=np.hstack([np.ones((len(recs),1)),surf,oh])
print(f"{len(recs)} solutions | {(y=='B').mean():.3f} Type-B")

In [ ]:
import time
def forward_capture(problem, steps, want_hidden):
    text = problem + "".join(f" {s} ки\n" for s in steps)
    ids = tok.encode(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model(ids, use_cache=False)
    tagmask = (ids[0] == TAG)
    n_tags = int(tagmask.sum())
    # step scores = P(good) at tag positions
    logit = out.logits[0][:, CAND].float()
    scores = torch.softmax(logit, -1)[:,0][tagmask].cpu().numpy()
    hid = None
    if want_hidden:
        hid = {}
        for li in LAYERS:
            h = out.hidden_states[li][0][tagmask]            # (n_tags, 4096) at step tags
            hid[li] = h.float().mean(0).cpu().numpy()         # solution-level = mean over step tags
    return scores, hid, n_tags

In [ ]:
# STAGE A — capture (solution-level activations per layer) + baseline step scores
t0=time.time(); acts={li:[] for li in LAYERS}; step_scores=[]; keep=[]; skipped=0
for i,r in enumerate(recs):
    prob = meta.get(r['id'],{}).get('problem','')
    try:
        sc, hid, nt = forward_capture(prob, r['steps_text'], want_hidden=True)
    except Exception as e:
        skipped+=1; continue
    if nt != len(r['steps_text']) or not np.isfinite(sc).all():   # truncation or fp16 NaN
        skipped+=1; continue
    step_scores.append(sc); keep.append(i)
    for li in LAYERS: acts[li].append(hid[li])
    if (i+1)%200==0: print(f"  {i+1}/{len(recs)} ({(time.time()-t0)/60:.1f}m, skipped {skipped})", flush=True)
keep=np.array(keep)
for li in LAYERS: acts[li]=np.array(acts[li])
np.savez(f"{WORK}/m2_acts.npz", keep=keep, **{f"L{li}":acts[li] for li in LAYERS})
import pickle; pickle.dump(step_scores, open(f"{WORK}/m2_scores.pkl","wb"))
print(f"captured {len(keep)} solutions ({skipped} skipped) in {(time.time()-t0)/60:.1f}m")
if skipped> len(recs)*0.15: print("WARNING: high skip rate — check fp16 NaNs / truncation")

In [ ]:
# STAGE B — probe each layer for difficulty; the 'where does difficulty live' figure
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
Lk, NSk = L[keep], NS[keep]
r2_len, r2_ns = [], []
for li in LAYERS:
    r2_len.append(cross_val_score(Ridge(1.0), acts[li], Lk, cv=5, scoring='r2').mean())
    r2_ns.append(cross_val_score(Ridge(1.0), acts[li], NSk, cv=5, scoring='r2').mean())
    print(f"  layer {li:2d}: log_length R2={r2_len[-1]:.3f}  n_steps R2={r2_ns[-1]:.3f}")
peak = LAYERS[int(np.argmax(r2_len))]
print(f"\npeak length-probe layer = {peak} (R2={max(r2_len):.3f})")
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.plot(LAYERS,r2_len,marker='o',label='log_length'); plt.plot(LAYERS,r2_ns,marker='s',label='n_steps')
plt.axvline(peak,ls='--',c='gray'); plt.xlabel('residual-stream layer'); plt.ylabel('probe R² (5-fold)')
plt.title('M2: where difficulty lives in Math-Shepherd-7B'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{WORK}/m2_where.png",dpi=140); plt.show()

In [ ]:
# Baseline (fp16, THIS setup) raw/ctl f1_B + step-label gate — the reference for ablation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
yk=y[keep]; CFk=CF[keep]
def split(n,s=0):
    rng=np.random.RandomState(s);idx=np.arange(n);rng.shuffle(idx);c=int(.8*n);return idx[:c],idx[c:]
def chain_min(ss): return np.array([1-ss[i].min() for i in range(len(ss))])
def f1_at(vec):
    tr,va=split(len(vec)); clf=LogisticRegression(max_iter=2000).fit(vec[tr].reshape(-1,1),yk[tr])
    return f1_score(yk[va],clf.predict(vec[va].reshape(-1,1)),pos_label='B')
def gate(ss):
    fs=[];fl=[]
    for i,idx in enumerate(keep):
        for s,l in zip(ss[i],recs[idx]['step_labels']):
            if l>=0: fs.append(1-s); fl.append(int(l))
    return roc_auc_score(fl,fs) if len(set(fl))>1 else float('nan')
base_v=chain_min(step_scores)
tr,_=split(len(base_v)); beta=np.linalg.lstsq(CFk[tr],base_v[tr],rcond=None)[0]
BASE={'raw_f1':f1_at(base_v),'ctl_f1':f1_at(base_v-CFk@beta),'gate':gate(step_scores)}
print("BASELINE (fp16):", {k:round(v,3) for k,v in BASE.items()})

In [ ]:
# STAGE C — MEAN-ablate the length direction at the peak layer, re-score, re-measure
w_np = Ridge(1.0).fit(acts[peak], Lk).coef_; w_np = w_np/np.linalg.norm(w_np)
mu = float((acts[peak] @ w_np).mean())                 # mean projection (for mean-ablation)
# hook the decoder layer whose OUTPUT is hidden_states[peak]  (hidden_states[0]=embeddings)
assert peak>=1, "peak at embeddings — unexpected; inspect Stage B"
target_layer = model.model.layers[peak-1]
W = torch.tensor(w_np, dtype=torch.float16)
def ablate_hook(mod, inp, out):
    h = out[0] if isinstance(out,tuple) else out
    wv = W.to(h.device)
    proj = (h.float() @ wv.float())                    # (B,L)
    h = h - ((proj - mu).to(h.dtype)).unsqueeze(-1) * wv   # set component to its mean
    return (h,)+tuple(out[1:]) if isinstance(out,tuple) else h
hk = target_layer.register_forward_hook(ablate_hook)
try:
    abl_scores=[]
    for j,idx in enumerate(keep):
        r=recs[idx]; prob=meta.get(r['id'],{}).get('problem','')
        sc,_,nt=forward_capture(prob, r['steps_text'], want_hidden=False)
        abl_scores.append(sc if (nt==len(r['steps_text']) and np.isfinite(sc).all()) else step_scores[j])
        if (j+1)%200==0: print(f"  ablated {j+1}/{len(keep)}", flush=True)
finally:
    hk.remove()
abl_v=chain_min(abl_scores)
tr,_=split(len(abl_v)); beta2=np.linalg.lstsq(CFk[tr],abl_v[tr],rcond=None)[0]
ABL={'raw_f1':f1_at(abl_v),'ctl_f1':f1_at(abl_v-CFk@beta2),'gate':gate(abl_scores)}
print("ABLATED  (fp16):", {k:round(v,3) for k,v in ABL.items()})

In [ ]:
# Decisive comparison
print("="*64)
print(f"  M2 — length-direction ablation at layer {peak} (fp16, T4x2, no quant)")
print("="*64)
print(f"  {'':<10}{'raw f1_B':>10}{'ctl f1_B':>10}{'step gate':>11}")
print(f"  {'baseline':<10}{BASE['raw_f1']:>10.3f}{BASE['ctl_f1']:>10.3f}{BASE['gate']:>11.3f}")
print(f"  {'ablated':<10}{ABL['raw_f1']:>10.3f}{ABL['ctl_f1']:>10.3f}{ABL['gate']:>11.3f}")
print(f"  {'Δ':<10}{ABL['raw_f1']-BASE['raw_f1']:>+10.3f}{ABL['ctl_f1']-BASE['ctl_f1']:>+10.3f}{ABL['gate']-BASE['gate']:>+11.3f}")
print("="*64)
drop_raw = BASE['raw_f1']-ABL['raw_f1']; drop_gate = BASE['gate']-ABL['gate']
if drop_raw>0.10 and drop_gate<0.05:
    print("  BEST CASE: raw f1_B falls toward controlled while the step gate is preserved ->")
    print("  difficulty is a localizable, removable direction. Causal + statistical control agree.")
elif drop_raw>0.10:
    print("  raw f1_B falls BUT the step gate also drops -> the direction is entangled with")
    print("  genuine competence; you removed capability, not only confound. Report as such.")
else:
    print("  raw f1_B barely moves -> difficulty is DISTRIBUTED, not a single direction. Real")
    print("  negative: confound control cannot be replaced by a targeted single-direction edit.")